In [52]:
import torch
import torch.nn as nn
import math
from pathlib import Path

In [61]:
# =========================================================
# 1. DATA
# =========================================================

text = "The cat sat on the mat because it was tired. After the cat rested for a while, the animal walked to the window and watched the birds outside. The birds flew over the garden, while the tired cat quietly followed them with its eyes."

text_old = "the cat sat"


chars = sorted(set(text))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

print("stoi:", stoi)
print("itos:", itos)


data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)

stoi: {' ': 0, ',': 1, '.': 2, 'A': 3, 'T': 4, 'a': 5, 'b': 6, 'c': 7, 'd': 8, 'e': 9, 'f': 10, 'g': 11, 'h': 12, 'i': 13, 'k': 14, 'l': 15, 'm': 16, 'n': 17, 'o': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'y': 26}
itos: {0: ' ', 1: ',', 2: '.', 3: 'A', 4: 'T', 5: 'a', 6: 'b', 7: 'c', 8: 'd', 9: 'e', 10: 'f', 11: 'g', 12: 'h', 13: 'i', 14: 'k', 15: 'l', 16: 'm', 17: 'n', 18: 'o', 19: 'q', 20: 'r', 21: 's', 22: 't', 23: 'u', 24: 'v', 25: 'w', 26: 'y'}


In [62]:
# =========================================================
# 2. TRAINING DATA
# =========================================================

BLOCK_SIZE = 4
D_MODEL = 128
VOCAB_SIZE = len(chars)

data = torch.tensor([stoi[x] for x in text])
print(data.size())


index = 5
print("".join(itos[x.item()] for x in data[index: index + BLOCK_SIZE]) + "-> " + "".join(itos[data[index + BLOCK_SIZE].item()]))

X =  []
Y =  []

for i in range(len(data) - BLOCK_SIZE):
    X.append( data[i: i + BLOCK_SIZE] )
    Y.append(data[i + BLOCK_SIZE])

X = torch.stack(X)
Y = torch.stack(Y)


print("vocab_size:", VOCAB_SIZE)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

torch.Size([230])
at s-> a
vocab_size: 27
X shape: torch.Size([226, 4])
Y shape: torch.Size([226])


In [63]:
# =========================================================
# 3. POSITIONAL ENCODING
# =========================================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        # Even dimensions
        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # [1, max_len, d_model]
        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        # x:
        # [batch, sequence, d_model]

        seq_len = x.shape[1]

        return x + self.pe[:, :seq_len]

In [64]:
# =========================================================
# 4. SELF-ATTENTION
# =========================================================

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        # NOTE: no embedding here.
        # The input is already [batch, sequence, d_model]
        # (embedded + position-encoded by CharModel).

        self.Wq = nn.Linear(
            d_model,
            d_model
        )

        self.Wk = nn.Linear(
            d_model,
            d_model
        )

        self.Wv = nn.Linear(
            d_model,
            d_model
        )

    def forward(self, x):

        # x: [batch, sequence, d_model]

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = Q @ K.transpose(-2, -1)

        scores = scores / math.sqrt(
            Q.shape[-1]
        )

        T = x.shape[1]

        mask = torch.tril(
            torch.ones(T, T, device=x.device)
        )

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        weights = torch.softmax(
            scores,
            dim=-1
        )

        output = weights @ V

        return output


In [65]:
# =========================================================
# 5. CHARACTER MODEL
# =========================================================

class CharModel(nn.Module):

    def __init__(self):

        super().__init__()

        # Character embedding
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        self.position = PositionalEncoding(
            D_MODEL,
            BLOCK_SIZE
        )

        # Self-attention
        self.attention = SelfAttention(
            D_MODEL
        )

        # Output layer
        self.fc = nn.Linear(
            D_MODEL,
            VOCAB_SIZE
        )

    def forward(self, x):

        # -----------------------------------------
        # Character embedding
        # -----------------------------------------

        x = self.embedding(x)

        # [batch, sequence, 64]

        # -----------------------------------------
        # Add position
        # -----------------------------------------

        x = self.position(x)

        # -----------------------------------------
        # Self-attention
        # -----------------------------------------

        x = self.attention(x)

        # -----------------------------------------
        # Predict next character
        # -----------------------------------------

        logits = self.fc(x)

        return logits


model = CharModel()

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Trainable parameters: 56475


In [66]:
# =========================================================
# 6. TRAINING
# =========================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

for step in range(3000):

    # [batch, sequence, vocab]
    logits = model(X)

    # Y holds ONE next character per block,
    # so only the last position is predicted.
    logits = logits[:, -1, :]

    loss = nn.functional.cross_entropy(
        logits,
        Y
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if step % 200 == 0:

        print(
            "step:",
            step,
            "loss:",
            loss.item()
        )


step: 0 loss: 3.382319450378418
step: 200 loss: 0.7842041254043579
step: 400 loss: 0.44407933950424194
step: 600 loss: 0.3177758753299713
step: 800 loss: 0.26205912232398987
step: 1000 loss: 0.8954828977584839
step: 1200 loss: 0.23903729021549225
step: 1400 loss: 0.22868752479553223
step: 1600 loss: 0.224160298705101
step: 1800 loss: 0.2173018902540207
step: 2000 loss: 0.21826200187206268
step: 2200 loss: 0.20047339797019958
step: 2400 loss: 0.19729529321193695
step: 2600 loss: 0.19559991359710693
step: 2800 loss: 0.1944754719734192


In [71]:
# =========================================================
# 7. GENERATION
# =========================================================

model.eval()

torch.manual_seed(1337)

context = torch.tensor([[stoi[c] for c in "anim"]])

result = "anim"


with torch.no_grad():

    for _ in range(30):

        logits = model(context)

        # Last position
        logits = logits[:, -1, :]

        # Probability
        probs = torch.softmax(
            logits,
            dim=-1
        )

        # Sample
        next_char = torch.multinomial(
            probs,
            num_samples=1
        )

        result += itos[next_char.item()]

        context = torch.cat([context, next_char], dim=1)[:, -BLOCK_SIZE:]

print("\nGenerated:")
print(result)


Generated:
animal walked them with its eyes.i
